# 03 - Expiry Risk Model
Outputs are **inventory decision-support estimates**, not medical or regulatory determinations.

Labels come from the ACTUAL unsold share of a batch at expiry (Low <15%, Medium 15-50%, High >=50%). Features use only information available at the snapshot date.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import config
from IPython.display import Image, display, Markdown
from src import expiry_prediction as ep
from src.feature_engineering import build_demand_features
import seaborn as sns, matplotlib.pyplot as plt

In [ ]:
sales = pd.read_csv(config.PROCESSED_DIR/'sales_daily_clean.csv', parse_dates=['Date'])
meds = pd.read_csv(config.PROCESSED_DIR/'medicines_clean.csv')
feat, _ = build_demand_features(sales, meds)
snaps, cat_map = ep.build_snapshots(sales, meds, feat)   # ~1 minute
snaps[['Medicine_ID','Snapshot_Date','days_until_expiry','current_stock','predicted_daily_demand','expected_unsold_ratio','actual_unsold_ratio','Risk_Label']].head()

In [ ]:
snaps['Risk_Label'].value_counts()

In [ ]:
res = ep.train_expiry_models(snaps, cat_map, save=False, verbose=False)
res['metrics'][res['metrics'].Split=='test'].round(3)

In [ ]:
labels = config.RISK_LABELS
for name, cm in res['confusion'].items():
    plt.figure(figsize=(4,3)); sns.heatmap(pd.DataFrame(cm,index=labels,columns=labels),annot=True,fmt='d',cmap='Blues'); plt.title(name); plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.show()

**Reading the results honestly:** compare the ML models with the *rule baseline* row. If they score about the same, the simple rule is just as good on this synthetic data.